# Find enriched SAE features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/feature_enrichment.ipynb)

Compare a positive sequence set with a length-matched background, inspect enriched features,
and export a signature for SAE-guided GRPO. The example uses nucleolar IDRs.
Enrichment describes this comparison; it does not establish a feature's biological function.

A GPU is recommended. In Colab, select **Runtime → Change runtime type → GPU** before running.
`DEVICE = "auto"` falls back to CPU. The background download and encoding take most of the work.

In [ ]:
# Install into this notebook's Python environment if IDiom is unavailable.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/rotskoff-group/idiom.git",
    ])

from huggingface_hub import hf_hub_download
import idiom


def example_data(name: str) -> str:
    """Download a cookbook example and return its cached local path."""
    return hf_hub_download(
        "jxliu2/idiom-data", f"example_data/{name}", repo_type="dataset",
    )


print("IDiom loaded from", idiom.__file__)

## Parameters

Start with a small subset to check the workflow. Increase `MAX_POSITIVE` and `MAX_BACKGROUND`
for a larger analysis; the demonstration defaults do not reproduce the released signatures.
Set either input path to use your own FASTA. Outputs are saved under `OUT_DIR`.

In [ ]:
from pathlib import Path

POSITIVE_FASTA = None  # None downloads the nucleolus example.
BACKGROUND_FASTA = None  # None downloads the held-out validation split.
NAME = "nucleolus"
SAE = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
OUT_DIR = Path("feature_enrichment_outputs")
MAX_POSITIVE = 128  # None uses all valid positive records.
MAX_BACKGROUND = 512  # Increase for a larger analysis.
BATCH_SIZE = 4
TOP_N = 30  # Maximum signature size; fewer features may pass the thresholds.
CASE = f"top{TOP_N}"
SEED = 0
N_FEATURES = 6
N_WINDOWS = 60
HALF_WIDTH = 7

In [ ]:
from pathlib import Path

import numpy as np

from idiom import IDiomSAE
from idiom.sae.features import FeatureDataset
from idiom.sae.features.enrichment import (
    FDR_ALPHA,
    LOG2OR_FLOOR,
    PREV_POS_FLOOR,
    enrich,
    enriched_mask,
    feature_counts,
    length_match,
    load_sequences,
    top_features,
    write_signature,
)

out = OUT_DIR
out.mkdir(parents=True, exist_ok=True)
if MAX_BACKGROUND < 1 or TOP_N < 1 or BATCH_SIZE < 1:
    raise ValueError("MAX_BACKGROUND, TOP_N, and BATCH_SIZE must be positive.")

sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
print(f"SAE: layer {sae.layer} of {sae.host_model}, {sae.sae.num_latents} latents")

## 1. Prepare the two sequence sets

Use canonical amino acids and `_IDR_x-y` headers with 1-based inclusive spans. `load_sequences`
treats missing or unusable spans as the whole sequence, so check annotations on full proteins.
This walkthrough uses an SAE trained on unprompted IDRs; flanks are excluded.

In [ ]:
if sae.fim_mode != "unprompted" or sae.region != "idr":
    raise ValueError("This walkthrough requires an SAE trained on unprompted IDRs.")

positive_path = Path(POSITIVE_FASTA) if POSITIVE_FASTA is not None else Path(
    example_data("protgps/nucleolus.fasta")
)
max_idr_length = sae.model.cfg.max_seq_len - 4  # START and three FIM markers


def usable_records(path):
    records = load_sequences(path)
    kept = [r for r in records if 0 < r.idr_end - r.idr_start <= max_idr_length]
    print(f"{Path(path).name}: kept {len(kept)} of {len(records)} canonical records within context")
    return kept


positives = usable_records(positive_path)
if MAX_POSITIVE is not None:
    if MAX_POSITIVE < 1:
        raise ValueError("MAX_POSITIVE must be positive or None.")
    if len(positives) > MAX_POSITIVE:
        chosen = np.random.default_rng(SEED).choice(len(positives), MAX_POSITIVE, replace=False)
        positives = [positives[i] for i in sorted(chosen)]
if not positives:
    raise ValueError("No usable positive records; check the FASTA and IDR spans.")
print(f"Positive set: {len(positives)} sequences")

The default background is downloaded from the held-out validation split. Only a sampled subset
is encoded. Exact IDR matches to the selected positive set are excluded.

Length matching reduces enrichment caused by different length distributions. Matching is
approximate when the pool cannot fill a length bin; inspect the distributions below.

In [ ]:
bg_path = Path(BACKGROUND_FASTA) if BACKGROUND_FASTA is not None else Path(hf_hub_download(
    "jxliu2/idiom-data", "training_sequences/validation.fasta", repo_type="dataset",
))
positive_idrs = {r.full_seq[r.idr_start:r.idr_end] for r in positives}
background_pool = [
    r for r in usable_records(bg_path)
    if r.full_seq[r.idr_start:r.idr_end] not in positive_idrs
]
background = length_match(
    positives, background_pool, n=MAX_BACKGROUND, rng=np.random.default_rng(SEED),
)
if not background:
    raise ValueError("No usable background records remain; choose another background FASTA.")
print(f"Background: sampled {len(background)} from {len(background_pool)} records")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
lengths = [[r.idr_end - r.idr_start for r in group] for group in (positives, background)]
bins = np.histogram_bin_edges(lengths[0] + lengths[1], bins=20)
for values, label in zip(lengths, ("Positive", "Background")):
    ax.hist(values, bins=bins, density=True, histtype="step", label=label)
ax.set(xlabel="IDR length (residues)", ylabel="Density")
ax.legend(frameon=False)
plt.show()

## 2. Build feature datasets

Encode both sets once and reuse the saved sparse activations for counts and sequence logos.
Reduce `BATCH_SIZE` if GPU memory is limited. Rerunning this cell replaces the dataset files.

In [ ]:
pos_fd = sae.build_feature_dataset(positives, out / "fd_positive", batch_size=BATCH_SIZE)
bg_fd = sae.build_feature_dataset(background, out / "fd_background", batch_size=BATCH_SIZE)
print("done")

## 3. Compare feature prevalence

Here a feature is counted once per sequence if it appears among the stored top-k feature IDs.
`enrich` computes smoothed log2 odds ratios and a standardized count statistic under a
hypergeometric null, then uses normal-approximation p-values with Benjamini–Hochberg correction.
Only features with sufficient pooled counts are tested.

In [ ]:
a, n_pos = feature_counts(pos_fd)
b, n_neg = feature_counts(bg_fd)
result = enrich(a, n_pos, b, n_neg, sae.sae.num_latents)
mask = enriched_mask(result)

print(f"{int(mask.sum())} enriched features")
print(f"  FDR < {FDR_ALPHA}, log2 odds ratio >= {LOG2OR_FLOOR}, prevalence >= {PREV_POS_FLOOR:.0%}")

In [ ]:
import matplotlib.pyplot as plt

active = result["active"]
x, y = result["log2or"][active], np.abs(result["z"][active])
enr = mask[active]

fig, ax = plt.subplots(figsize=(4.6, 3.6), constrained_layout=True)
ax.scatter(x[~enr], y[~enr], s=4, alpha=0.25, lw=0, color="#c3ced0", label="other")
ax.scatter(x[enr], y[enr], s=8, alpha=0.9, lw=0, color="#c1440e", label="enriched")
ax.axvline(LOG2OR_FLOOR, ls="--", lw=0.8, color="#110d1b")
ax.axvline(0, lw=0.8, color="#110d1b")
ax.set_xlabel("log$_2$ odds ratio")
ax.set_ylabel("|z|")
ax.set_title(f"{NAME}: {int(mask.sum())} enriched features", fontsize=10)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 4. Export a feature signature

Rank passing features by log2 odds ratio and remove features whose strongest background
activations cluster near IDR boundaries. This is a heuristic for boundary-associated features,
not proof that the remaining features represent motifs.

The signature contains at most `TOP_N` IDs. If none pass, no signature is written; inspect the
inputs or increase the sample sizes. Rerunning replaces the named case in the signature file.

In [ ]:
ids = top_features(result, n=TOP_N, drop_boundary=True, feature_dir=bg_fd)
sig_path = None
if ids:
    sig_path = write_signature(
        out / "signature.json", {NAME: ids}, case=CASE,
        provenance={
            "sae": SAE, "positive": str(positive_path), "background": str(bg_path),
            "n_pos": int(n_pos), "n_background": int(n_neg), "seed": SEED,
            "length_matched": True, "exact_positive_idrs_excluded": True,
            "boundary_dropped": True, "rank": "log2 odds ratio, descending",
        },
    )
    print(f"Saved {len(ids)} features to {sig_path.resolve()}: {ids}")
else:
    print("No features passed the filters. No signature was written in this run.")

## 5. Inspect sequence windows

For each selected feature, take one window around the strongest positive activation per sequence
and plot an information-content logo. Windows near sequence ends shift to fit, so their peaks
are not always centered. Logos summarize the selected windows; they do not establish function.

In [ ]:
import logomaker

positive_dataset = FeatureDataset(pos_fd)
show = ids[:N_FEATURES]
if not show:
    print("No signature features to plot.")
else:
    fig, axes = plt.subplots(
        len(show), 1, figsize=(7, 2 * len(show)), constrained_layout=True, squeeze=False,
    )
    window_length = 2 * HALF_WIDTH + 1
    for ax, feature_id in zip(axes[:, 0], show):
        sequence_ids, _ = positive_dataset.top_sequences(
            feature_id, n=positive_dataset.n_seqs, sort_by="peak",
        )
        windows = []
        for seq_id in sequence_ids:
            positions, values = positive_dataset.trace(int(seq_id), feature_id)
            if len(positions) < window_length or values.max() <= 0:
                continue
            fim_string = positive_dataset.sequence(int(seq_id))
            residues = "".join(fim_string[int(p)] for p in positions)
            peak = int(values.argmax())
            start = min(max(peak - HALF_WIDTH, 0), len(residues) - window_length)
            windows.append(residues[start:start + window_length])
            if len(windows) >= N_WINDOWS:
                break
        if len(windows) < 2:
            ax.set_axis_off()
            ax.set_title(f"Feature {feature_id}: too few windows")
            continue
        logomaker.Logo(
            logomaker.alignment_to_matrix(windows, to_type="information"),
            ax=ax, color_scheme="chemistry",
        )
        ax.set(title=f"Feature {feature_id} ({len(windows)} windows)", ylabel="Bits")
        ax.set_xticks([])
    plt.show()

## Next: use the signature for GRPO

Download `signature.json` from the Colab file browser, or use its local path. In
[`cookbook/scripts/grpo/sae_features.bash`](https://github.com/rotskoff-group/idiom/blob/main/cookbook/scripts/grpo/sae_features.bash),
set `FEATURES` to that path and set `SIGNATURE` and `CASE` as printed below. Use the same SAE
that produced the signature when configuring its reward. The script provides the full training
configuration, including entropy and length objectives.

Training is a separate GPU job; this notebook stops after analysis and export.

In [ ]:
if sig_path is not None:
    print(f"Signature file: {sig_path.resolve()}")
    print(f"SIGNATURE={NAME}")
    print(f"CASE={CASE}")
    print(f"SAE used for feature IDs: {SAE}")
else:
    print("No signature exported in this run; review the enrichment results above.")